# Module 3, Notebook 2: Factor Number Selection

## Learning Objectives

By completing this notebook, you will:

1. **Implement** Bai-Ng information criteria (IC1, IC2, IC3)
2. **Apply** scree plots and eigenvalue ratios for visual diagnostics
3. **Understand** the bias-variance trade-off in factor selection
4. **Compare** different selection methods on simulated data
5. **Validate** consistency of IC criteria across sample sizes
6. **Select** optimal number of factors for real applications

## Prerequisites

- Module 3, Notebook 1: Stock-Watson estimation
- Understanding of model selection (AIC/BIC)
- Eigenvalue decomposition

## Estimated Time: 75 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Bai-Ng information criteria (IC1, IC2, IC3)",
    "scree plots and eigenvalue ratios for visual diagnostics",
    "the bias-variance trade-off in factor selection",
    "different selection methods on simulated data",
    "consistency of IC criteria across sample sizes",
    "optimal number of factors for real applications"
])

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import eigh
from typing import Dict, Tuple

# Set random seed
np.random.seed(123)

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
%matplotlib inline

print("Setup complete!")

## 1. The Factor Number Selection Problem

### Why Selection Matters

Choosing the number of factors $r$ is critical:

- **Too few factors (r too small):** Underfit the common variation, missing important factors
- **Too many factors (r too large):** Overfit, capturing idiosyncratic noise as "factors"

### The Bias-Variance Trade-off

This is analogous to polynomial degree selection in regression:

```
Error
  |
  |     Bias²
  |    /
  |   /        \
  |  /          \  Total Error
  | /            \
  |/              \
  |                \___
  |          Variance  \___
  +-----|-----|-----|-----|--> r
      r=1   r=2   r*  r=5
```

Information criteria formalize this trade-off by penalizing complexity.

### Bai-Ng Framework

General form:
$$IC(r) = \ln[V(r)] + r \cdot g(N, T)$$

where:
- $V(r)$ = residual variance with $r$ factors (fit measure)
- $g(N,T)$ = penalty function (complexity measure)

**Select:** $\hat{r} = \arg\min_{r} IC(r)$

## 2. Implementation: Bai-Ng Information Criteria

We'll implement three variants that differ in their penalty strength.

In [ ]:
class BaiNgSelector:
    """
    Bai-Ng information criteria for selecting number of factors.
    
    Parameters
    ----------
    r_max : int
        Maximum number of factors to consider
    standardize : bool, default=True
        Standardize variables before analysis
    """
    
    def __init__(self, r_max=10, standardize=True):
        self.r_max = r_max
        self.standardize = standardize
        
        # Results storage
        self.ic_values_ = None
        self.pc_values_ = None
        self.eigenvalues_ = None
        self.V_r_ = None
        
    def fit(self, X):
        """
        Compute information criteria for r = 0, 1, ..., r_max.
        
        Parameters
        ----------
        X : ndarray (T, N)
            Panel data
            
        Returns
        -------
        self
        """
        X = np.asarray(X)
        T, N = X.shape
        
        if self.r_max >= min(N, T):
            raise ValueError(f"r_max must be < min(N,T), got {self.r_max}")
        
        # Step 1: Standardize
        if self.standardize:
            X_mean = np.mean(X, axis=0)
            X_std = np.std(X, axis=0, ddof=1)
            X_std[X_std < 1e-10] = 1.0
            X_proc = (X - X_mean) / X_std
        else:
            X_proc = X - np.mean(X, axis=0)
        
        # Step 2: Compute eigenvalues
        Sigma_X = X_proc.T @ X_proc / T
        eigenvalues, _ = eigh(Sigma_X)
        eigenvalues = np.sort(eigenvalues)[::-1]  # Descending
        self.eigenvalues_ = eigenvalues
        
        # Step 3: Compute V(r) for each r
        self.V_r_ = self._compute_residual_variance(eigenvalues, N)
        
        # Step 4: Compute IC criteria
        self.ic_values_ = self._compute_ic_criteria(T, N)
        
        # Step 5: Compute PC criteria (alternative formulation)
        self.pc_values_ = self._compute_pc_criteria(T, N)
        
        return self
    
    def _compute_residual_variance(self, eigenvalues, N):
        """
        Compute residual variance V(r) = 1 - sum(top r eigenvalues) / N.
        
        For standardized data, total variance = N (each variable has var=1).
        """
        V_r = np.zeros(self.r_max + 1)
        V_r[0] = 1.0  # No factors: all variance unexplained
        
        cumsum_eigenvalues = np.cumsum(eigenvalues)
        for r in range(1, self.r_max + 1):
            # Proportion of variance NOT explained by r factors
            V_r[r] = 1.0 - cumsum_eigenvalues[r-1] / N
        
        return V_r
    
    def _compute_ic_criteria(self, T, N):
        """
        Compute IC1, IC2, IC3 criteria.
        
        IC_p(r) = ln[V(r)] + r * g_p(N, T)
        """
        NT = N * T
        C_NT = min(np.sqrt(N), np.sqrt(T))
        
        ic_values = {
            'IC1': np.zeros(self.r_max + 1),
            'IC2': np.zeros(self.r_max + 1),
            'IC3': np.zeros(self.r_max + 1),
        }
        
        for r in range(self.r_max + 1):
            V_r = self.V_r_[r]
            
            # Prevent log(0)
            if V_r < 1e-12:
                V_r = 1e-12
            
            log_V = np.log(V_r)
            
            # IC1: Balanced penalty
            penalty_1 = r * ((N + T) / NT) * np.log(NT / (N + T))
            ic_values['IC1'][r] = log_V + penalty_1
            
            # IC2: Weaker penalty (tends to select more factors)
            penalty_2 = r * ((N + T) / NT) * np.log(C_NT**2)
            ic_values['IC2'][r] = log_V + penalty_2
            
            # IC3: Stronger penalty (tends to select fewer factors)
            penalty_3 = r * np.log(C_NT**2) / (C_NT**2)
            ic_values['IC3'][r] = log_V + penalty_3
        
        return ic_values
    
    def _compute_pc_criteria(self, T, N):
        """
        Compute PC1, PC2, PC3 criteria.
        
        PC_p(r) = V(r) + r * sigma_hat^2 * g_p(N, T)
        """
        NT = N * T
        C_NT = min(np.sqrt(N), np.sqrt(T))
        
        # Estimate idiosyncratic variance using r_max factors
        sigma_sq_hat = self.V_r_[self.r_max]
        
        pc_values = {
            'PC1': np.zeros(self.r_max + 1),
            'PC2': np.zeros(self.r_max + 1),
            'PC3': np.zeros(self.r_max + 1),
        }
        
        for r in range(self.r_max + 1):
            V_r = self.V_r_[r]
            
            # PC1
            penalty_1 = r * sigma_sq_hat * ((N + T) / NT) * np.log(NT / (N + T))
            pc_values['PC1'][r] = V_r + penalty_1
            
            # PC2
            penalty_2 = r * sigma_sq_hat * ((N + T) / NT) * np.log(C_NT**2)
            pc_values['PC2'][r] = V_r + penalty_2
            
            # PC3
            penalty_3 = r * sigma_sq_hat * np.log(C_NT**2) / (C_NT**2)
            pc_values['PC3'][r] = V_r + penalty_3
        
        return pc_values
    
    def select(self, criterion='IC1'):
        """
        Select number of factors by minimizing criterion.
        
        Parameters
        ----------
        criterion : str
            One of 'IC1', 'IC2', 'IC3', 'PC1', 'PC2', 'PC3'
            
        Returns
        -------
        r_hat : int
            Selected number of factors
        """
        if self.ic_values_ is None:
            raise ValueError("Must call fit() first")
        
        if criterion in self.ic_values_:
            values = self.ic_values_[criterion]
        elif criterion in self.pc_values_:
            values = self.pc_values_[criterion]
        else:
            raise ValueError(f"Unknown criterion: {criterion}")
        
        r_hat = np.argmin(values)
        return r_hat
    
    def select_all(self):
        """
        Select using all six criteria.
        
        Returns
        -------
        selections : dict
            Dictionary of criterion -> selected r
        """
        selections = {}
        for name in ['IC1', 'IC2', 'IC3', 'PC1', 'PC2', 'PC3']:
            selections[name] = self.select(name)
        return selections


# Test implementation
print("BaiNgSelector class created successfully!")

## 3. Simulation Study: Validation

Let's verify that the criteria correctly identify the true number of factors in simulated data.

In [ ]:
def simulate_dfm_known_factors(T=300, N=80, r_true=4, noise_ratio=0.5, seed=None):
    """
    Simulate DFM with known number of factors.
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Generate loadings
    Lambda = np.random.randn(N, r_true) * 0.8
    
    # Generate factors with mild persistence
    F = np.zeros((T, r_true))
    F[0] = np.random.randn(r_true)
    for t in range(1, T):
        F[t] = 0.5 * F[t-1] + np.random.randn(r_true) * 0.7
    
    # Generate observations
    e = np.random.randn(T, N) * noise_ratio
    X = F @ Lambda.T + e
    
    return X, r_true


# Generate data with r=4 true factors
print("Simulating data with r=4 true factors...\n")
X, r_true = simulate_dfm_known_factors(T=300, N=80, r_true=4, noise_ratio=0.5, seed=42)
print(f"Data shape: {X.shape}")
print(f"True number of factors: {r_true}")

# Fit selector
print("\nFitting Bai-Ng selector...")
selector = BaiNgSelector(r_max=12, standardize=True)
selector.fit(X)

# Get selections
selections = selector.select_all()

print("\n" + "="*60)
print("FACTOR NUMBER SELECTION RESULTS")
print("="*60)
print(f"\nTrue number of factors: {r_true}\n")

print("Information Criteria (IC):")
for name in ['IC1', 'IC2', 'IC3']:
    r_hat = selections[name]
    match = "✓" if r_hat == r_true else "✗"
    print(f"  {name}: r = {r_hat:2d}  {match}")

print("\nPenalized Criteria (PC):")
for name in ['PC1', 'PC2', 'PC3']:
    r_hat = selections[name]
    match = "✓" if r_hat == r_true else "✗"
    print(f"  {name}: r = {r_hat:2d}  {match}")

print("="*60)

### Visualization: All Criteria

In [ ]:
# Purpose: Visualize how each criterion varies with r

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

r_values = np.arange(selector.r_max + 1)

# IC criteria (top row)
for i, (name, ax) in enumerate(zip(['IC1', 'IC2', 'IC3'], axes[0])):
    values = selector.ic_values_[name]
    r_hat = selections[name]
    
    ax.plot(r_values, values, 'o-', linewidth=2, markersize=7)
    ax.axvline(r_hat, color='red', linestyle='--', linewidth=2.5,
               label=f'Selected: r={r_hat}', alpha=0.8)
    ax.axvline(r_true, color='green', linestyle=':', linewidth=2.5,
               label=f'True: r={r_true}', alpha=0.8)
    ax.set_xlabel('Number of factors (r)', fontsize=11)
    ax.set_ylabel(f'{name} Value', fontsize=11)
    ax.set_title(f'{name} Criterion', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# PC criteria (bottom row)
for i, (name, ax) in enumerate(zip(['PC1', 'PC2', 'PC3'], axes[1])):
    values = selector.pc_values_[name]
    r_hat = selections[name]
    
    ax.plot(r_values, values, 'o-', linewidth=2, markersize=7, color='darkorange')
    ax.axvline(r_hat, color='red', linestyle='--', linewidth=2.5,
               label=f'Selected: r={r_hat}', alpha=0.8)
    ax.axvline(r_true, color='green', linestyle=':', linewidth=2.5,
               label=f'True: r={r_true}', alpha=0.8)
    ax.set_xlabel('Number of factors (r)', fontsize=11)
    ax.set_ylabel(f'{name} Value', fontsize=11)
    ax.set_title(f'{name} Criterion', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNote the clear minimum at (or near) r=4 for all criteria!")

## 4. Visual Diagnostics: Scree Plots

Scree plots show eigenvalue decay and help identify the "elbow" where factors become weak.

In [ ]:
def plot_scree_diagnostics(selector, r_true=None, n_show=15):
    """
    Create comprehensive scree plot diagnostics.
    
    Parameters
    ----------
    selector : BaiNgSelector
        Fitted selector
    r_true : int, optional
        True number of factors (if known)
    n_show : int
        Number of eigenvalues to display
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    eigenvalues = selector.eigenvalues_[:n_show]
    idx = np.arange(1, n_show + 1)
    
    # 1. Scree plot
    ax = axes[0, 0]
    ax.plot(idx, eigenvalues, 'o-', linewidth=2.5, markersize=8)
    
    r_ic1 = selector.select('IC1')
    if r_ic1 < n_show:
        ax.axvline(r_ic1, color='red', linestyle='--', linewidth=2,
                   label=f'IC1: r={r_ic1}', alpha=0.7)
    if r_true is not None and r_true < n_show:
        ax.axvline(r_true, color='green', linestyle=':', linewidth=2,
                   label=f'True: r={r_true}', alpha=0.7)
    
    ax.set_xlabel('Factor Number', fontsize=12)
    ax.set_ylabel('Eigenvalue', fontsize=12)
    ax.set_title('Scree Plot', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 2. Log eigenvalues (better for small values)
    ax = axes[0, 1]
    ax.plot(idx, np.log(eigenvalues), 'o-', linewidth=2.5, markersize=8, color='purple')
    
    if r_ic1 < n_show:
        ax.axvline(r_ic1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    if r_true is not None and r_true < n_show:
        ax.axvline(r_true, color='green', linestyle=':', linewidth=2, alpha=0.7)
    
    ax.set_xlabel('Factor Number', fontsize=12)
    ax.set_ylabel('Log Eigenvalue', fontsize=12)
    ax.set_title('Log Scree Plot', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # 3. Eigenvalue ratios
    ax = axes[1, 0]
    ratios = eigenvalues[:-1] / eigenvalues[1:]
    ax.plot(idx[:-1], ratios, 'o-', linewidth=2.5, markersize=8, color='orange')
    ax.axhline(2.0, color='gray', linestyle=':', linewidth=2,
               label='Ratio = 2.0', alpha=0.6)
    
    ax.set_xlabel('Factor Number', fontsize=12)
    ax.set_ylabel('Eigenvalue Ratio ($d_r / d_{r+1}$)', fontsize=12)
    ax.set_title('Eigenvalue Ratios', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 4. Cumulative variance explained
    ax = axes[1, 1]
    cumvar = np.cumsum(selector.eigenvalues_) / selector.eigenvalues_.sum()
    ax.plot(idx, cumvar[:n_show], 'o-', linewidth=2.5, markersize=8, color='green')
    ax.axhline(0.8, color='gray', linestyle=':', linewidth=2,
               label='80% threshold', alpha=0.6)
    
    if r_ic1 < n_show:
        ax.axvline(r_ic1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    if r_true is not None and r_true < n_show:
        ax.axvline(r_true, color='green', linestyle=':', linewidth=2, alpha=0.7)
    
    ax.set_xlabel('Number of Factors', fontsize=12)
    ax.set_ylabel('Cumulative Variance Explained', fontsize=12)
    ax.set_title('Cumulative Variance', fontsize=13, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\nVariance Explained Summary:")
    for r in [1, 2, 3, 4, 5, 6, 8, 10]:
        if r <= len(cumvar):
            print(f"  r = {r:2d}: {cumvar[r-1]:.1%}")


# Generate diagnostic plots
plot_scree_diagnostics(selector, r_true=4, n_show=15)

The scree plot shows a clear "elbow" at r=4, confirming the IC criteria selection!

## Exercise 4.1: Monte Carlo Consistency Study

Verify that IC criteria become more accurate as sample size increases.

**Task:** Run 100 simulations for different (N, T) combinations and compute the fraction of times each criterion correctly selects r=4.

In [ ]:
# YOUR CODE HERE
# ---------------

def monte_carlo_consistency(n_sims=100, sample_sizes=None, r_true=4):
    """
    Monte Carlo study of IC criteria consistency.
    
    Parameters
    ----------
    n_sims : int
        Number of Monte Carlo replications
    sample_sizes : list of tuples
        List of (T, N) pairs to test
    r_true : int
        True number of factors
        
    Returns
    -------
    results : DataFrame
        Success rates for each criterion and sample size
    """
    if sample_sizes is None:
        sample_sizes = [(100, 50), (200, 80), (400, 120)]
    
    # TODO: Initialize storage for results
    results = None  # Replace this
    
    # TODO: Loop over sample sizes
    for T, N in sample_sizes:
        # TODO: Run n_sims simulations
        # For each simulation:
        #   1. Generate data with simulate_dfm_known_factors
        #   2. Fit BaiNgSelector
        #   3. Check if each criterion selects r_true
        #   4. Accumulate success counts
        
        pass  # Replace with your implementation
    
    # TODO: Compute success rates
    # TODO: Create visualization of results
    
    return results


# Test your implementation
# results = monte_carlo_consistency(n_sims=100)

<details>
<summary>Hint 1</summary>
Use a nested loop: outer loop over sample_sizes, inner loop over n_sims. For each simulation, generate data and check if selector.select(criterion) == r_true.
</details>

<details>
<summary>Hint 2</summary>
Store results in a dictionary of lists, then convert to DataFrame. Success rate = (number of correct selections) / n_sims.
</details>

In [ ]:
# AUTO-GRADED TESTS
# ------------------

def test_exercise_4_1():
    """Test Monte Carlo consistency study."""
    # Run with small number of simulations for testing
    results = monte_carlo_consistency(n_sims=20, 
                                      sample_sizes=[(100, 50), (200, 80)],
                                      r_true=4)
    
    # Test 1: Results not None
    assert results is not None, "Don't forget to return results!"
    
    # Test 2: Correct structure
    assert isinstance(results, pd.DataFrame), "Results should be a DataFrame"
    assert len(results) == 2, "Should have 2 rows (one per sample size)"
    
    # Test 3: Success rates in valid range
    for col in ['IC1', 'IC2', 'IC3']:
        if col in results.columns:
            assert results[col].min() >= 0, f"{col} success rate should be >= 0"
            assert results[col].max() <= 1, f"{col} success rate should be <= 1"
    
    # Test 4: Generally improves with sample size
    # (Not strict due to randomness, but check at least one improves)
    any_improved = any(results.iloc[1][col] >= results.iloc[0][col] 
                       for col in ['IC1', 'IC2', 'IC3'] if col in results.columns)
    assert any_improved, "At least one criterion should improve with larger sample"
    
    print("✅ Exercise 4.1 passed!")
    print("\nResults:")
    print(results)

# Uncomment to test:
# test_exercise_4_1()

### Solution to Exercise 4.1

In [ ]:
# SOLUTION
# --------

def monte_carlo_consistency(n_sims=100, sample_sizes=None, r_true=4):
    """
    Monte Carlo study of IC criteria consistency.
    """
    if sample_sizes is None:
        sample_sizes = [(100, 50), (200, 80), (400, 120)]
    
    # Initialize storage
    results_dict = {
        'T': [], 'N': [],
        'IC1': [], 'IC2': [], 'IC3': [],
        'PC1': [], 'PC2': [], 'PC3': []
    }
    
    print(f"Running {n_sims} simulations for each sample size...\n")
    
    for T, N in sample_sizes:
        print(f"Sample size: T={T}, N={N}")
        
        # Track successes for each criterion
        successes = {name: 0 for name in ['IC1', 'IC2', 'IC3', 'PC1', 'PC2', 'PC3']}
        
        for sim in range(n_sims):
            # Generate data
            X_sim, _ = simulate_dfm_known_factors(T=T, N=N, r_true=r_true,
                                                   noise_ratio=0.5, seed=sim)
            
            # Fit selector
            sel = BaiNgSelector(r_max=min(10, N//8), standardize=True)
            sel.fit(X_sim)
            
            # Check each criterion
            selections = sel.select_all()
            for name, r_hat in selections.items():
                if r_hat == r_true:
                    successes[name] += 1
        
        # Compute success rates
        results_dict['T'].append(T)
        results_dict['N'].append(N)
        for name in ['IC1', 'IC2', 'IC3', 'PC1', 'PC2', 'PC3']:
            rate = successes[name] / n_sims
            results_dict[name].append(rate)
            print(f"  {name}: {rate:.1%}")
        print()
    
    # Convert to DataFrame
    results = pd.DataFrame(results_dict)
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x_labels = [f"T={T}, N={N}" for T, N in sample_sizes]
    x_pos = np.arange(len(sample_sizes))
    
    width = 0.12
    colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']
    
    for i, (name, color) in enumerate(zip(['IC1', 'IC2', 'IC3', 'PC1', 'PC2', 'PC3'], colors)):
        offset = width * (i - 2.5)
        ax.bar(x_pos + offset, results[name], width, label=name, alpha=0.8, color=color)
    
    ax.axhline(1.0, color='black', linestyle=':', linewidth=1.5, alpha=0.5)
    ax.set_xlabel('Sample Size', fontsize=12)
    ax.set_ylabel('Success Rate (Correct Selection)', fontsize=12)
    ax.set_title(f'IC Criteria Consistency (True r={r_true}, {n_sims} sims)',
                fontsize=14, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels)
    ax.set_ylim([0, 1.1])
    ax.legend(ncol=6, fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    return results


# Run simulation study
print("Running Monte Carlo consistency study...\n")
results_mc = monte_carlo_consistency(n_sims=50,
                                     sample_sizes=[(100, 50), (200, 80), (400, 120)],
                                     r_true=4)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(results_mc.to_string(index=False))
print("\nNote: Success rates generally increase with sample size,")
print("      confirming asymptotic consistency!")

## Exercise 4.2: Penalty Comparison

Investigate how different penalty strengths affect selection behavior.

**Task:** For a fixed dataset, plot how the three IC penalties differ and explain which tends to select more/fewer factors.

In [ ]:
# YOUR CODE HERE
# ---------------

def compare_penalty_functions(T=300, N=80):
    """
    Visualize how IC penalty functions differ.
    
    Parameters
    ----------
    T, N : int
        Sample dimensions
        
    Returns
    -------
    penalty_df : DataFrame
        Penalty values for r = 1, ..., 10
    """
    # TODO: Compute penalty functions for r = 1 to 10
    # Formulas:
    #   g_1(r) = r * (N+T)/(NT) * log(NT/(N+T))
    #   g_2(r) = r * (N+T)/(NT) * log(min(N,T))
    #   g_3(r) = r * log(min(N,T)) / min(N,T)
    
    penalty_df = None  # Replace this
    
    # TODO: Create visualization comparing the three penalties
    
    return penalty_df


# Test your implementation
# penalty_df = compare_penalty_functions(T=300, N=80)

<details>
<summary>Hint</summary>
Compute C_NT = min(sqrt(N), sqrt(T)) first. Then use the formulas above for each r. Plot all three on the same axes to see differences.
</details>

In [ ]:
# AUTO-GRADED TESTS
# ------------------

def test_exercise_4_2():
    """Test penalty comparison."""
    penalty_df = compare_penalty_functions(T=300, N=80)
    
    assert penalty_df is not None, "Don't forget to return penalty_df!"
    assert 'IC1' in penalty_df.columns, "Should include IC1 penalty"
    assert 'IC2' in penalty_df.columns, "Should include IC2 penalty"
    assert 'IC3' in penalty_df.columns, "Should include IC3 penalty"
    
    # IC3 should have strongest penalty (largest values)
    assert penalty_df['IC3'].mean() > penalty_df['IC1'].mean(), \
        "IC3 should have stronger penalty than IC1"
    
    print("✅ Exercise 4.2 passed!")
    print("\nPenalty comparison:")
    print(penalty_df.head(10))

# Uncomment to test:
# test_exercise_4_2()

### Solution to Exercise 4.2

In [ ]:
# SOLUTION
# --------

def compare_penalty_functions(T=300, N=80):
    """
    Visualize how IC penalty functions differ.
    """
    NT = N * T
    C_NT = min(np.sqrt(N), np.sqrt(T))
    
    r_vals = np.arange(1, 11)
    
    # Compute penalties
    penalties = {
        'r': r_vals,
        'IC1': r_vals * ((N + T) / NT) * np.log(NT / (N + T)),
        'IC2': r_vals * ((N + T) / NT) * np.log(C_NT**2),
        'IC3': r_vals * np.log(C_NT**2) / (C_NT**2)
    }
    
    penalty_df = pd.DataFrame(penalties)
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Absolute penalty values
    ax1.plot(r_vals, penalty_df['IC1'], 'o-', linewidth=2.5, 
            markersize=8, label='IC1 (balanced)', alpha=0.8)
    ax1.plot(r_vals, penalty_df['IC2'], 's-', linewidth=2.5,
            markersize=8, label='IC2 (weak)', alpha=0.8)
    ax1.plot(r_vals, penalty_df['IC3'], '^-', linewidth=2.5,
            markersize=8, label='IC3 (strong)', alpha=0.8)
    ax1.set_xlabel('Number of Factors (r)', fontsize=12)
    ax1.set_ylabel('Penalty Value', fontsize=12)
    ax1.set_title('Penalty Functions', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Penalty per factor
    ax2.plot(r_vals, penalty_df['IC1'] / r_vals, 'o-', linewidth=2.5,
            markersize=8, label='IC1', alpha=0.8)
    ax2.plot(r_vals, penalty_df['IC2'] / r_vals, 's-', linewidth=2.5,
            markersize=8, label='IC2', alpha=0.8)
    ax2.plot(r_vals, penalty_df['IC3'] / r_vals, '^-', linewidth=2.5,
            markersize=8, label='IC3', alpha=0.8)
    ax2.set_xlabel('Number of Factors (r)', fontsize=12)
    ax2.set_ylabel('Penalty per Factor', fontsize=12)
    ax2.set_title('Per-Factor Penalty', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nFor T={T}, N={N}:")
    print(f"  C_NT = min(sqrt(N), sqrt(T)) = {C_NT:.2f}")
    print(f"\nPenalty ordering: IC2 < IC1 < IC3")
    print(f"  => IC2 selects MOST factors (weak penalty)")
    print(f"  => IC3 selects FEWEST factors (strong penalty)")
    print(f"  => IC1 is balanced compromise")
    
    return penalty_df


# Run comparison
penalty_df = compare_penalty_functions(T=300, N=80)
print("\nPenalty values for r=1 to 10:")
print(penalty_df.to_string(index=False))

## Summary

### Key Takeaways

1. **Factor selection is a bias-variance trade-off:** Too few factors underfit, too many overfit

2. **IC criteria provide data-driven selection:** Bai-Ng penalties calibrated for factor model asymptotics

3. **Three penalty strengths available:**
   - IC2: Weak penalty, selects more factors
   - IC1: Balanced, recommended for most applications
   - IC3: Strong penalty, conservative selection

4. **Visual diagnostics complement formal tests:** Scree plots and eigenvalue ratios aid interpretation

5. **Consistency holds asymptotically:** All criteria correctly select r_0 as N, T → ∞

6. **Practical recommendation:** Report IC1, IC2, IC3 and verify agreement

### Decision Framework

When selecting factors:

1. **Start with IC1** (balanced)
2. **Check IC2 and IC3** for robustness
3. **Inspect scree plot** for visual confirmation
4. **Consider application:** Forecasting may prefer fewer factors (IC3), data description more (IC2)
5. **Verify interpretability:** Do selected factors make economic sense?

### What's Next

In Module 4, we'll explore **maximum likelihood estimation** via the EM algorithm, which provides an alternative to PCA-based methods with better small-sample properties and rigorous uncertainty quantification.

### Additional Resources

- Bai & Ng (2002): "Determining the Number of Factors in Approximate Factor Models"
- Onatski (2010): "Determining the Number of Factors from Empirical Distribution of Eigenvalues"
- Module guide: `guides/02_factor_number_selection.md`

---

**Completion:** Module 3 complete! Proceed to Module 4 for likelihood-based estimation methods.